In [1]:
!pip install dotenv llama_index llama-index-embeddings-huggingface coloredlogs -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.

In [2]:
!nvidia-smi

Wed Oct  1 18:47:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import logging

fl = logging.getLogger("filelock")
fl.handlers = [logging.NullHandler()]
fl.propagate = False
fl.setLevel(logging.CRITICAL + 1)  # higher than CRITICAL => nothing emits

In [4]:
import torch

# Check if CUDA (the GPU interface) is available to PyTorch
is_available = torch.cuda.is_available()

print(f"Is CUDA available? {is_available}")

if is_available:
    # Get the number of GPUs
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs available: {gpu_count}")
    
    # Get the name of the current GPU
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Name: {gpu_name}")
else:
    print("CUDA is not available. PyTorch is running on the CPU.")

Is CUDA available? True
Number of GPUs available: 1
GPU Name: Tesla P100-PCIE-16GB


In [5]:
# Forcefully remove the old directory if it exists
!rm -rf /kaggle/working/AlphaAgent

In [6]:
# Import the Kaggle secrets client
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Retrieve the secret token
# Make sure the string 'GITHUB_TOKEN' matches the Label you used in Kaggle Secrets
personal_token = user_secrets.get_secret("AlphaAgent_Read")

# Construct the clone URL with authentication
# Replace 'your-username' with your actual GitHub username
!git clone https://{personal_token}@github.com/AdityaSreevatsaK/AlphaAgent.git

# Navigate into the cloned directory
%cd AlphaAgent

Cloning into 'AlphaAgent'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (305/305), done.
remote: Compressing objects: 100% (211/211), done.
remote: Total 305 (delta 171), reused 218 (delta 86), pack-reused 0 (from 0)
Receiving objects: 100% (305/305), 302.05 KiB | 4.38 MiB/s, done.
Resolving deltas: 100% (171/171), done.
/kaggle/working/AlphaAgent


In [7]:
import coloredlogs
import logging
import sys
from pathlib import Path

fl = logging.getLogger("filelock")
fl.disabled = True  

# Set root logger
logging.getLogger().setLevel(logging.WARNING)

# Suppress noisy loggers
for noisy in ["urllib3", "huggingface_hub", "httpx", "fsspec"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)


# Add your project's src directory to the Python path
sys.path.append(str(Path.cwd() / 'src'))

from utils.hub_manager import get_from_hub
from utils.config import get_config

# Get the repository ID from your config file
huggingface_config = get_config("huggingface")
repo_id = huggingface_config.get("repo_id")

if repo_id:
    print(f"Downloading data from Hugging Face repository: {repo_id}")
    # This function will download the 'processed' folder from your HF repo
    get_from_hub(repo_id=repo_id, local_dir_name="processed", repo_type="dataset")
    print("Data download complete.")
else:
    print("Hugging Face repo_id not found in config.yml.")

2025-10-01 18:47:06 - root - INFO - Data not found locally. Downloading from Hugging Face Hub: AdityaSreevatsaK/AlphaAgent-Data


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 80 files:   0%|          | 0/80 [00:00<?, ?it/s]

full-submission.txt: 0.00B [00:00, ?B/s]

data/processed/filings/0000019617/10-K/0(…):   0%|          | 0.00/1.39M [00:00<?, ?B/s]

data/processed/filings/0000019617/10-K/0(…):   0%|          | 0.00/1.38M [00:00<?, ?B/s]

... (74 outputs left out)

TM_news.txt: 0.00B [00:00, ?B/s]

TSLA_news.txt: 0.00B [00:00, ?B/s]

2025-10-01 18:47:09 - root - INFO - Successfully downloaded data to '/kaggle/working/AlphaAgent/data/processed'.


Data download complete.


In [8]:
# This command will list all files and folders recursively (-R) 
# within the specified directory, showing you the complete structure.
!find /kaggle/working/AlphaAgent/data/processed/ -type d

/kaggle/working/AlphaAgent/data/processed/
/kaggle/working/AlphaAgent/data/processed/filings
/kaggle/working/AlphaAgent/data/processed/filings/0001318605
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K/0000950170-22-000796
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K/0001564590-21-004599
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K/0001628280-24-002390
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K/0000950170-23-001409
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/10-K/0001628280-25-003063
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/8-K
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/8-K/0001628280-25-033842
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/8-K/0001628280-25-026148
/kaggle/working/AlphaAgent/data/processed/filings/0001318605/8-K/0001628280-25-034692
/kaggle/working/AlphaA

In [9]:
# The -m flag runs the script as a module from the current directory,
# ensuring that all imports from 'src' work correctly.
import logging

# Silence the overly verbose loggers from the libraries
# This tells the logger to only show messages that are WARNING level or higher for these specific libraries
logging.getLogger("llama_index.core.node_parser.node_utils").setLevel(logging.WARNING)
logging.getLogger("urllib3.connectionpool").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
logging.getLogger("filelock").setLevel(logging.WARNING)

print("--- Set verbose loggers to WARNING level. Starting ingestion... ---")

!python -m scripts.run_rag_ingestion

--- Set verbose loggers to WARNING level. Starting ingestion... ---
2025-10-01 18:47:26.667004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759344446.857087      97 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759344446.912155      97 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-01 18:47:43 - root - INFO - Starting RAG data ingestion process from: /kaggle/working/AlphaAgent/data/processed
2025-10-01 18:47:43 - root - INFO - Using device: cuda
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 818kB/s]
README.md: 10.5kB [00:00, 33.1MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 215kB